# Offline ALNS Repair Model Training Pipeline

This notebook documents the complete training workflow for learning a repair policy in the Hybrid ALNS solver using offline supervised learning.

## Overview

The pipeline trains a GradientBoosting classifier to rank feasible bin repair actions during ALNS search. The workflow follows three main stages:
1. **Baseline training** on synthetic instances
2. **Collection** of realistic states from actual ALNS trajectories
3. **Augmented retraining** on synthetic + collected data

Dataset generation and model training are **separate steps**. Raw datasets are saved as
`.pkl` files under `training_data/` so they can be reused across training runs without
regenerating from scratch.

All model artefacts are saved as pickle files for deployment in the solver.

## 1. Problem Definition

At ALNS step $t$, a partial solution state is $x_t$. For each feasible bin assignment action $a \in \mathcal{A}(x_t)$, the model predicts a score used to rank actions. The optimizer then favors the highest-ranked repair actions.

**Key insight**: We optimize for **ranking quality** (ROC-AUC / Average Precision), not just binary classification accuracy, because ALNS uses model scores to prioritize candidate bins.

**Deliverables**:
- `training_data/synthetic_v1.pkl` — baseline synthetic dataset
- `repair_model_v1.pkl` — baseline model trained on synthetic data
- `training_data/alns_states_v1.pkl` — collected states from real ALNS trajectories
- `training_data/synthetic_v2.pkl` — smaller synthetic dataset for augmented run
- `repair_model_v2.pkl` — augmented model trained on merged data

## 2. Data Augmentation Strategy

Synthetic data alone under-represents states encountered during real ALNS search. To reduce this distribution mismatch, we collect additional realistic states and retrain on the merged dataset.

**Key parameters**:
- `instances`: total number of synthetic instances to generate
- `n_min`, `n_max`: instance size range
- `max_negatives`: negative sampling budget (class balance control)
- `iterations`: ALNS trajectory length per collection run
- `seed`: reproducibility

Dataset files are saved to `training_data/` and can be reused independently of training.

In [ ]:
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
    hybrid_alns_solver,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.models import (
    load_repair_model,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training import (
    collect_alns_states,
    generate_dataset,
    train_repair_model,
)
from bin_packing_optimization.utilities.benchmarking import Benchmark, create_benchmark

## 3. Training Pipeline

### Step 1a: Generate Baseline Synthetic Dataset

Sample 4,000 synthetic instances, label bin placements with the BFD oracle, and save
the raw dataset to `training_data/synthetic_v1.pkl`.

**Expected output**: `training_data/synthetic_v1.pkl`

In [ ]:
generate_dataset(
    instances=4000,
    n_min=50,
    n_max=200,
    max_negatives=5,
    seed=0,
    workers=1,
    output="training_data/synthetic_v1.pkl",
)

#### Execution & Results

**Dataset generation**
- 4,000 synthetic instances
- 1,313,891 rows, 11 features
- Positive rate: 0.2194

### Step 1b: Train Baseline Model

Train on the generated synthetic dataset using 5-fold cross-validation.

**Expected output**: `repair_model_v1.pkl`

In [ ]:
train_repair_model(
    data=["training_data/synthetic_v1.pkl"],
    output="repair_model_v1.pkl",
    cv_folds=5,
)

#### Execution & Results

**Training**
- GradientBoosting (180 estimators, depth 4, lr 0.05, subsample 0.8)
- Duration: 1,696.0s
- 5-fold CV ROC-AUC: 0.8769 ± 0.0003

**Test metrics**
- ROC-AUC: 0.8790
- Average Precision: 0.7306
- F1: 0.6459
- Precision: 0.5530
- Recall: 0.7763

### Step 2: Collect Real ALNS States

Use the baseline model to collect realistic repair states from actual ALNS trajectories.

**Expected output**: `training_data/alns_states_v1.pkl`

In [ ]:
collect_alns_states(
    model_path="repair_model_v1.pkl",
    instances=500,
    n_min=50,
    n_max=200,
    iterations=200,
    max_negatives=5,
    seed=1,
    output="training_data/alns_states_v1.pkl",
)

#### Execution & Results

**Collection summary**
- 500 ALNS runs
- 309,539 collected rows
- Positive rate: 0.297

**Output**: `training_data/alns_states_v1.pkl`

This augmented dataset is large enough to meaningfully shift training toward real search states.

### Step 3a: Generate Synthetic Dataset for Augmented Run

Generate a smaller synthetic dataset (2,000 instances) to be merged with the collected
ALNS states. Using fewer synthetic instances here because the ALNS data already provides
substantial coverage of real search states.

**Expected output**: `training_data/synthetic_v2.pkl`

In [ ]:
generate_dataset(
    instances=2000,
    n_min=50,
    n_max=200,
    max_negatives=3,
    seed=0,
    workers=4,
    output="training_data/synthetic_v2.pkl",
)

#### Execution & Results

**Dataset generation**
- 2,000 synthetic instances
- Positive rate: ~0.25 (fewer negatives per positive with max_negatives=3)

### Step 3b: Augmented Model Retraining

Merge the second synthetic dataset with collected ALNS states and retrain.
Both dataset paths are passed via `data`; they are loaded and concatenated automatically.

**Expected output**: `repair_model_v2.pkl` (final model for deployment)

In [ ]:
train_repair_model(
    data=[
        "training_data/synthetic_v2.pkl",
        "training_data/alns_states_v1.pkl",
    ],
    output="repair_model_v2.pkl",
    cv_folds=3,
    no_plots=True,
    no_learning_curves=True,
)

#### Execution & Results

**Merged training data**
- Augmented with 309,539 ALNS rows
- Effective dataset after processing: 566,381 rows, 11 features
- Positive rate: 0.4036 (expected min: 0.2500)
- Class 0 (negatives): 337,772 samples
- Class 1 (positives): 228,609 samples

**Training**
- GradientBoosting (500 estimators, depth 6, lr 0.03, subsample 0.75)
- Duration: 3,638.4s
- 3-fold CV ROC-AUC: 0.9024 ± 0.0011

**Test metrics**
- ROC-AUC: 0.9067
- Average Precision: 0.8915
- F1: 0.7759
- Precision: 0.8174
- Recall: 0.7385

**Confusion matrix**
- TP: 25,324 | FP: 5,657
- FN: 8,968  | TN: 45,009

**Summary**: Ranking quality improved significantly from v1 to v2 (0.8790 → 0.9067 ROC-AUC), confirming augmentation strategy effectiveness.

## 4. Model Performance Comparison

| Metric | v1 (Baseline) | v2 (Augmented) | Improvement |
|--------|--------------|----------------|------------|
| **ROC-AUC** | 0.8790 | 0.9067 | +3.2% |
| **Avg Precision** | 0.7306 | 0.8915 | +22.1% |
| **F1-Score** | 0.6459 | 0.7759 | +20.1% |
| **Precision** | 0.5530 | 0.8174 | +47.8% |
| **Recall** | 0.7763 | 0.7385 | +4.9% |
| **Training Samples** | 1,313,891 | 566,381* | — |

\* v2 used only 2,000 synthetic instances augmented with 309,539 collected ALNS states, resulting in 566,381 effective samples after balancing.

**Conclusion**: The augmentation strategy significantly improves ranking quality (primary metric for ALNS deployment) while using fewer total training samples. Model v2 is preferred for deployment.

## 5. Quick Validation Benchmark

Spot-check the deployed model on `falkenauer-u` with default iteration budget.

In [ ]:
model_bundle = load_repair_model("repair_model_v2.pkl")

benchmark: Benchmark = create_benchmark(
    dataset_key="falkenauer-u",
    solver_module=hybrid_alns_solver,
)
benchmark.run(method=None, method_args={"model_bundle": model_bundle})
benchmark.save_results_to_csv()